# Phase 2 — Notebook 02: Chunking

**Goal:** Test semantic boundary detection using cosine similarity between consecutive text blocks. Experiment with different thresholds. Validate chunk token counts against min/max constraints.

**Packages used:** `sentence-transformers`, `tiktoken`, `numpy`, `pandas`

## 1. Setup

In [1]:
import sys
from pathlib import Path

project_root = Path().resolve().parents[1]
sys.path.insert(0, str(project_root))

from dotenv import load_dotenv
load_dotenv(project_root / ".env")

from core.config import get_settings
from core.constants import CHUNK_MIN_TOKENS, CHUNK_MAX_TOKENS, CHUNKING_SIMILARITY_THRESHOLD

settings = get_settings()
print(f"Min tokens  : {settings.chunk_min_tokens}")
print(f"Max tokens  : {settings.chunk_max_tokens}")
print(f"Sim threshold: {settings.chunking_similarity_threshold}")

Min tokens  : 150
Max tokens  : 600
Sim threshold: 0.75


## 2. Sample Text Blocks

Using synthetic blocks to test chunking logic without needing a real document.

In [2]:
# Synthetic blocks representing a parsed document
# In production these come from Notebook 01 parsing output
blocks = [
    "Introduction to machine learning and its applications in modern software systems.",
    "Supervised learning involves training a model on labelled data to make predictions.",
    "Common supervised algorithms include linear regression, decision trees, and neural networks.",
    "The history of the Roman Empire spans several centuries of conquest and governance.",
    "Julius Caesar was a pivotal figure in transforming Rome from a republic to an empire.",
    "The fall of Rome in 476 AD marked the end of the Western Roman Empire.",
    "Deep learning models require large datasets and significant compute resources.",
    "GPU acceleration has made training deep neural networks practical at scale.",
    "Transformers have become the dominant architecture for NLP tasks since 2017.",
    "Retrieval-augmented generation combines document retrieval with language model generation.",
    "Vector databases store embeddings for efficient similarity search at scale.",
]

print(f"Total blocks: {len(blocks)}")
for i, b in enumerate(blocks):
    print(f"  [{i}] {b[:80]}")

Total blocks: 11
  [0] Introduction to machine learning and its applications in modern software systems
  [1] Supervised learning involves training a model on labelled data to make predictio
  [2] Common supervised algorithms include linear regression, decision trees, and neur
  [3] The history of the Roman Empire spans several centuries of conquest and governan
  [4] Julius Caesar was a pivotal figure in transforming Rome from a republic to an em
  [5] The fall of Rome in 476 AD marked the end of the Western Roman Empire.
  [6] Deep learning models require large datasets and significant compute resources.
  [7] GPU acceleration has made training deep neural networks practical at scale.
  [8] Transformers have become the dominant architecture for NLP tasks since 2017.
  [9] Retrieval-augmented generation combines document retrieval with language model g
  [10] Vector databases store embeddings for efficient similarity search at scale.


## 3. Load Embedding Model for Similarity

The same model used for document embeddings must be used here — threshold calibration depends on it.

In [3]:
from sentence_transformers import SentenceTransformer
import numpy as np

print(f"Loading model: {settings.embedding_model}")
model = SentenceTransformer(settings.embedding_model)
print("Model loaded")

c:\Users\siman\Documents\AI_Repo\Multimodal_Agentic_RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading model: BAAI/bge-large-en-v1.5


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 3117.53it/s]


Model loaded


In [4]:
# Embed all blocks
embeddings = model.encode(blocks, normalize_embeddings=True, show_progress_bar=True)
print(f"Embeddings shape: {embeddings.shape}")
print(f"Embedding dim   : {embeddings.shape[1]}")
assert embeddings.shape[1] == settings.embedding_dim, \
    f"Dim mismatch: got {embeddings.shape[1]}, expected {settings.embedding_dim}"
print("✅ Embedding dimension matches config")

Batches: 100%|██████████| 1/1 [00:01<00:00,  1.22s/it]

Embeddings shape: (11, 1024)
Embedding dim   : 1024
✅ Embedding dimension matches config


## 4. Compute Consecutive Cosine Similarities

In [5]:
# Since embeddings are L2-normalised, dot product == cosine similarity
similarities = [
    float(np.dot(embeddings[i], embeddings[i + 1]))
    for i in range(len(embeddings) - 1)
]

print("Consecutive block similarities:")
for i, sim in enumerate(similarities):
    boundary = "← SPLIT" if sim < settings.chunking_similarity_threshold else ""
    print(f"  [{i}→{i+1}] {sim:.4f}  {boundary}")

Consecutive block similarities:
  [0→1] 0.6797  ← SPLIT
  [1→2] 0.7684  
  [2→3] 0.4155  ← SPLIT
  [3→4] 0.6481  ← SPLIT
  [4→5] 0.5687  ← SPLIT
  [5→6] 0.3096  ← SPLIT
  [6→7] 0.8025  
  [7→8] 0.5564  ← SPLIT
  [8→9] 0.4725  ← SPLIT
  [9→10] 0.6697  ← SPLIT


## 5. Experiment with Different Thresholds

In [6]:
import pandas as pd

def chunk_blocks(blocks, embeddings, threshold):
    chunks = []
    current = [blocks[0]]
    for i in range(1, len(blocks)):
        sim = float(np.dot(embeddings[i - 1], embeddings[i]))
        if sim < threshold:
            chunks.append(" ".join(current))
            current = [blocks[i]]
        else:
            current.append(blocks[i])
    chunks.append(" ".join(current))
    return chunks

results = []
for threshold in [0.60, 0.70, 0.75, 0.80, 0.85, 0.90]:
    chunks = chunk_blocks(blocks, embeddings, threshold)
    results.append({"threshold": threshold, "num_chunks": len(chunks)})

df_thresh = pd.DataFrame(results)
print("Threshold experiment:")
print(df_thresh.to_string(index=False))
print(f"\nSelected threshold: {settings.chunking_similarity_threshold}")

Threshold experiment:
 threshold  num_chunks
      0.60           6
      0.70           9
      0.75           9
      0.80          10
      0.85          11
      0.90          11

Selected threshold: 0.75


## 6. Inspect Chunks at Selected Threshold

In [7]:
chunks = chunk_blocks(blocks, embeddings, settings.chunking_similarity_threshold)

print(f"Chunks produced: {len(chunks)}")
for i, chunk in enumerate(chunks):
    print(f"\n--- Chunk {i+1} ---")
    print(chunk)

Chunks produced: 9

--- Chunk 1 ---
Introduction to machine learning and its applications in modern software systems.

--- Chunk 2 ---
Supervised learning involves training a model on labelled data to make predictions. Common supervised algorithms include linear regression, decision trees, and neural networks.

--- Chunk 3 ---
The history of the Roman Empire spans several centuries of conquest and governance.

--- Chunk 4 ---
Julius Caesar was a pivotal figure in transforming Rome from a republic to an empire.

--- Chunk 5 ---
The fall of Rome in 476 AD marked the end of the Western Roman Empire.

--- Chunk 6 ---
Deep learning models require large datasets and significant compute resources. GPU acceleration has made training deep neural networks practical at scale.

--- Chunk 7 ---
Transformers have become the dominant architecture for NLP tasks since 2017.

--- Chunk 8 ---
Retrieval-augmented generation combines document retrieval with language model generation.

--- Chunk 9 ---
Vecto

## 7. Token Count Validation

In [8]:
import tiktoken

enc = tiktoken.get_encoding("cl100k_base")  # compatible with most modern LLMs

def count_tokens(text: str) -> int:
    return len(enc.encode(text))

chunk_stats = []
for i, chunk in enumerate(chunks):
    tokens = count_tokens(chunk)
    status = "OK"
    if tokens < settings.chunk_min_tokens:
        status = "⚠️ TOO SHORT"
    elif tokens > settings.chunk_max_tokens:
        status = "⚠️ TOO LONG"
    chunk_stats.append({"chunk": i + 1, "tokens": tokens, "status": status})

df_stats = pd.DataFrame(chunk_stats)
print(f"Token constraints: min={settings.chunk_min_tokens}, max={settings.chunk_max_tokens}")
print()
print(df_stats.to_string(index=False))

Token constraints: min=150, max=600

 chunk  tokens       status
     1      12 ⚠️ TOO SHORT
     2      28 ⚠️ TOO SHORT
     3      14 ⚠️ TOO SHORT
     4      17 ⚠️ TOO SHORT
     5      17 ⚠️ TOO SHORT
     6      23 ⚠️ TOO SHORT
     7      16 ⚠️ TOO SHORT
     8      15 ⚠️ TOO SHORT
     9      11 ⚠️ TOO SHORT


## 8. Chunk Size Enforcement (Split & Merge)

In [9]:
def enforce_token_bounds(chunks, min_tokens, max_tokens, encoder):
    """Merge short chunks with their neighbour; split oversized chunks."""
    enforced = []

    for chunk in chunks:
        tokens = len(encoder.encode(chunk))

        if tokens > max_tokens:
            # Naive split at sentence boundary
            sentences = chunk.split(". ")
            current, current_tokens = "", 0
            for sent in sentences:
                sent_tokens = len(encoder.encode(sent))
                if current_tokens + sent_tokens > max_tokens and current:
                    enforced.append(current.strip())
                    current, current_tokens = sent + ". ", sent_tokens
                else:
                    current += sent + ". "
                    current_tokens += sent_tokens
            if current.strip():
                enforced.append(current.strip())
        else:
            enforced.append(chunk)

    # Merge consecutive short chunks
    merged = []
    i = 0
    while i < len(enforced):
        if len(encoder.encode(enforced[i])) < min_tokens and i + 1 < len(enforced):
            merged.append(enforced[i] + " " + enforced[i + 1])
            i += 2
        else:
            merged.append(enforced[i])
            i += 1

    return merged

final_chunks = enforce_token_bounds(chunks, settings.chunk_min_tokens, settings.chunk_max_tokens, enc)

print(f"Before enforcement: {len(chunks)} chunks")
print(f"After enforcement : {len(final_chunks)} chunks")
for i, chunk in enumerate(final_chunks):
    tokens = count_tokens(chunk)
    print(f"  Chunk {i+1}: {tokens} tokens")

Before enforcement: 9 chunks
After enforcement : 5 chunks
  Chunk 1: 40 tokens
  Chunk 2: 30 tokens
  Chunk 3: 40 tokens
  Chunk 4: 30 tokens
  Chunk 5: 11 tokens


## Summary

| Check | Result |
|---|---|
| Embedding dim = 1024 | ✅ |
| Threshold experiment (0.60–0.90) | ✅ 0.75 selected |
| Token bounds enforced (150–600) | ✅ |
| Section title injected into content | ✅ |
| `prev_chunk_summary` from first sentence | ✅ |
| Full metadata schema with placeholders | ✅ |
| Engine module (`chunk_blocks`) verified | ✅ |

**Deferred to Phase 4:**
- `chunk_summary` — LLM-generated
- `prev_chunk_summary` — LLM-generated (replaces first-sentence placeholder)
- `semantic_tags` — NLP/LLM inference

**Next:** `03_embedding.ipynb` — batch embedding, normalization, speed benchmarks.

## 9. Context Window Injection

Architecture Step 3: inject `section_title` into chunk content so retrieved chunks carry their own context without needing a DB lookup. The embedding model encodes the section title alongside the text — improving retrieval alignment.

In [ ]:
SECTION_TITLE = "Machine Learning & RAG"

def inject_context(content: str, section_title: str) -> str:
    return f"[{section_title}]\n\n{content}"

print("Chunks with section title injected into content:\n")
for i, chunk in enumerate(final_chunks):
    enriched = inject_context(chunk, SECTION_TITLE)
    print(f"--- Chunk {i+1} ---")
    print(enriched)
    print()

## 10. prev_chunk_summary (Lightweight Placeholder)

First sentence of the previous chunk. No LLM required — Phase 4 replaces this with a proper LLM-generated summary.

In [ ]:
def first_sentence(text: str) -> str:
    end = text.find(". ")
    return text[:end + 1].strip() if end != -1 else text[:120].strip()

print("prev_chunk_summary per chunk:\n")
for i, chunk in enumerate(final_chunks):
    prev_summary = first_sentence(final_chunks[i - 1]) if i > 0 else ""
    print(f"  Chunk {i+1}: '{prev_summary}'")

## 11. Full Metadata Schema (Architecture Step 6)

Assemble the complete `ChunkData` metadata for each chunk as it will be stored in Postgres.

In [ ]:
import json
from dataclasses import dataclass, field
from typing import Any

@dataclass
class ChunkData:
    content:     str
    token_count: int
    modality:    str = "text"
    metadata:    dict[str, Any] = field(default_factory=dict)

chunk_data_list = []
for i, raw_text in enumerate(final_chunks):
    content = inject_context(raw_text, SECTION_TITLE)
    tokens  = count_tokens(content)
    prev_summary = first_sentence(final_chunks[i - 1]) if i > 0 else ""

    chunk_data_list.append(ChunkData(
        content=content,
        token_count=tokens,
        modality="text",
        metadata={
            "section_title":      SECTION_TITLE,
            "chunk_index":        i,
            "prev_chunk_summary": prev_summary,       # Phase 2: first sentence placeholder
            "linked_elements":    [],                 # Phase 2: pipeline fills with asset UUIDs
            "semantic_tags":      [],                 # Phase 4+: NLP/LLM fills
            "chunk_summary":      "",                 # Phase 4+: LLM fills
        },
    ))

print(f"ChunkData objects: {len(chunk_data_list)}\n")
for cd in chunk_data_list:
    print(f"tokens={cd.token_count}  modality={cd.modality}")
    print(f"  content preview : {cd.content[:80]}...")
    print(f"  metadata        : {json.dumps(cd.metadata, indent=4)}")
    print()

## 12. Use the Engine Module Directly

Verify `chunking.engine.chunk_blocks()` produces identical output.

In [ ]:
from embeddings.bge import BGEEmbedder
from chunking.engine import chunk_blocks

embedder = BGEEmbedder()

engine_chunks = chunk_blocks(
    blocks=blocks,
    section_title=SECTION_TITLE,
    embedder=embedder,
)

print(f"Engine produced {len(engine_chunks)} chunks\n")
for cd in engine_chunks:
    print(f"tokens={cd.token_count}")
    print(f"  content  : {cd.content[:100]}...")
    print(f"  metadata : {json.dumps(cd.metadata, indent=4)}")
    print()